# 3. Status Codes

Every HTTP response has a **status code**. The status code is a quick signal about what happened.

- `200` range: success
- `300` range: redirect
- `400` range: client-side problem, such as a missing page
- `500` range: server-side problem


In [1]:
import requests

TIMEOUT = 10

## Ask for Specific Status Codes

httpbin is a public testing API. This endpoint lets us request a response with a specific status code.


In [2]:
status_codes = [200, 201, 204, 301, 400, 404, 500]

for code in status_codes:
    url = f"https://httpbin.org/status/{code}"
    response = requests.get(url, timeout=TIMEOUT, allow_redirects=False)
    print(code, "->", response.status_code, response.reason)

200 -> 200 OK
201 -> 201 CREATED
204 -> 204 NO CONTENT
301 -> 301 MOVED PERMANENTLY
400 -> 400 BAD REQUEST
404 -> 404 NOT FOUND
500 -> 500 INTERNAL SERVER ERROR


## Group Status Codes by Family

A status code's first digit tells us the general category.


In [3]:
def status_family(status_code):
    if 200 <= status_code < 300:
        return "success"
    if 300 <= status_code < 400:
        return "redirect"
    if 400 <= status_code < 500:
        return "client error"
    if 500 <= status_code < 600:
        return "server error"
    return "other"

for code in status_codes:
    print(code, status_family(code))

200 success
201 success
204 success
301 redirect
400 client error
404 client error
500 server error


## `response.ok`

`response.ok` is `True` for status codes below 400. That includes success and redirects.


In [4]:
good_response = requests.get("https://httpbin.org/status/200", timeout=TIMEOUT)
missing_response = requests.get("https://httpbin.org/status/404", timeout=TIMEOUT)

print(good_response.status_code, good_response.ok)
print(missing_response.status_code, missing_response.ok)

200 True
404 False


## `raise_for_status()`

`raise_for_status()` is useful when we want Python to stop or warn us after a bad status code.

Here we catch the error so the notebook keeps running.


In [5]:
response = requests.get("https://httpbin.org/status/404", timeout=TIMEOUT)

try:
    response.raise_for_status()
    print("Request worked")
except requests.HTTPError as error:
    print("Request did not work")
    print(error)

Request did not work
404 Client Error: NOT FOUND for url: https://httpbin.org/status/404


## Check Before Reading Data

Many error responses do not include useful JSON. It is safer to check the status code before trying to use the data.


In [6]:
response = requests.get("https://httpbin.org/status/404", timeout=TIMEOUT)

if response.ok:
    print(response.json())
else:
    print(f"Skipping JSON because the status code was {response.status_code}.")

Skipping JSON because the status code was 404.


## A Small Helper Function

A helper function can keep API code neat.


In [7]:
def describe_response(response):
    status = response.status_code
    family = status_family(status)
    print(f"{status} means {family}.")

    if response.ok:
        print("Python says response.ok is True.")
    else:
        print("Python says response.ok is False.")

for code in [200, 404, 500]:
    response = requests.get(f"https://httpbin.org/status/{code}", timeout=TIMEOUT)
    describe_response(response)
    print("---")

200 means success.
Python says response.ok is True.
---
404 means client error.
Python says response.ok is False.
---
500 means server error.
Python says response.ok is False.
---
